# Pitaco — fine-tuning a 4B student for Portuguese aspect-based sentiment

Kaggle version of the main experiment from [github.com/PedroMLegnaghi/pitaco](https://github.com/PedroMLegnaghi/pitaco). It rebuilds the data splits from the Olist dataset (attached as input), checks they match the split IDs committed in the repo, fine-tunes the student chosen in `STUDENT_CONFIG` (default: `Qwen/Qwen3-4B-Instruct-2507` with QLoRA — Qwen3.5 needs bfloat16, which the T4 lacks) on teacher-labeled (silver) reviews, and predicts the hand-labeled gold sets.

**Accelerator: GPU T4** (not P100 — too old for the Triton kernels used here).

In [ ]:
# Run settings — the only cell you need to change.
# Full run : STUDENT_CONFIG = 'configs/student/qwen3_4b_qlora.yaml'; PREDICT_LIMIT = None
# Test run : STUDENT_CONFIG = 'configs/student/kaggle_test_qwen3_4b_qlora.yaml'; PREDICT_LIMIT = 3
STUDENT_CONFIG = "configs/student/kaggle_test_qwen3_4b_qlora.yaml"
PREDICT_LIMIT = 3

In [ ]:
# Preflight: stop immediately (instead of failing 5 cells later) if a notebook setting is missing.
import glob
import socket
import subprocess

problems = []
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    problems.append(
        "Internet is OFF: right panel → Session options → Internet → On (needs a verified phone)"
    )
if not glob.glob("/kaggle/input/**/olist_order_reviews_dataset.csv", recursive=True):
    problems.append(
        'Olist dataset not attached: right panel → Input → + Add Input → search "olistbr brazilian-ecommerce" → +'
    )
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True
).stdout
if "T4" not in gpu:
    problems.append(
        f"GPU is not a T4 (got: {gpu.strip() or 'none'}): right panel → Session options → Accelerator → GPU T4 x2"
    )
assert not problems, "\n".join(problems)
print("preflight OK:", gpu.strip().splitlines()[0])

In [ ]:
import os
import subprocess

subprocess.run(
    ["git", "clone", "--depth", "1", "https://github.com/PedroMLegnaghi/pitaco.git"], check=True
)
os.chdir("pitaco")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r requirements/kaggle-gpu.txt
!pip install -q --no-deps --ignore-requires-python -e .
# Kaggle's torchaudio/torchcodec are built for its original torch; we upgraded torch and use neither.
!pip uninstall -y -q torchaudio torchcodec

In [ ]:
import subprocess


def run(*args):
    """Run a pitaco command; stop the notebook if it fails (plain `!` commands don't)."""
    print("$", " ".join(args), flush=True)
    subprocess.run(args, check=True)

Experiment tracking: add a Kaggle secret named `WANDB_API_KEY` (Add-ons → Secrets) to log to Weights & Biases; without it the run logs offline.

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    os.environ["WANDB_MODE"] = "offline"
    print("W&B secret not found: logging offline")

Rebuild the splits from the raw CSV and verify they are identical to the committed ones (`git diff` must be empty).

In [ ]:
import glob
import os
import shutil

found = glob.glob("/kaggle/input/**/olist_order_reviews_dataset.csv", recursive=True)
assert found, (
    'Olist not attached: right panel → Add Input → search "Brazilian E-Commerce Public Dataset by Olist"'
)
os.makedirs("data/raw", exist_ok=True)
shutil.copy(found[0], "data/raw/")
print("using", found[0])

In [ ]:
run("pitaco", "data", "--config", "configs/data.yaml")
run("git", "diff", "--stat", "--exit-code", "data/splits")
print("splits identical to the repo")

In [ ]:
run("pitaco", "train-student", "--config", STUDENT_CONFIG)

Predictions on gold dev/test: the base model zero-shot (before fine-tuning) and the fine-tuned adapter. Both measure batch-1 latency and batched throughput on the T4.

In [ ]:
from pathlib import Path

import yaml

OUT = yaml.safe_load(Path(STUDENT_CONFIG).read_text())["output_dir"]
LIMIT = ["--limit", str(PREDICT_LIMIT)] if PREDICT_LIMIT else []
run(
    "pitaco", "predict-student", "--config", STUDENT_CONFIG, *LIMIT
)  # base model, before fine-tuning
run("pitaco", "predict-student", "--config", STUDENT_CONFIG, "--adapter", f"{OUT}/adapter", *LIMIT)

In [ ]:
!tar czf /kaggle/working/pitaco_outputs.tgz {OUT}/adapter results/predictions results/efficiency
!ls -lh /kaggle/working